In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.optimize import least_squares

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [3]:
panel = pd.read_csv(INPUT / 'fit/data/panel.csv', index_col='dblp_id')
Q = panel.to_numpy(dtype=float).T
complete = Q[:,np.isfinite(Q).all(axis=0)]
models = {tag:np.load(INPUT / tag / 'trajectories.npy') for tag in CONFIG['models']}
sources = {'empirical_complete':complete, **models}
# simulation contract checked once here; empirical contract belongs upstream
assert all(a.shape == (Y+1, CONFIG['n_sim']) and np.isfinite(a).all() and (a>=0).all() for a in models.values())
metadata = pd.concat([pd.read_csv(INPUT / tag / 'metadata.csv') for tag in CONFIG['models']], ignore_index=True)
metadata.to_csv(OUTPUT / 'simulation_metadata.csv', index=False)

In [4]:
def gini(q):
    ordered = np.sort(q)
    n = len(ordered)
    return (2*np.dot(np.arange(1,n+1),ordered)/(n*ordered.sum())-(n+1)/n) if ordered.sum()>0 else np.nan
def correlation(x,y):
    return stats.spearmanr(x,y).statistic if len(x)>2 and np.ptp(x)>0 and np.ptp(y)>0 else np.nan

rows = []
for source, array in {'empirical_available':Q, **sources}.items():
    for t in range(Y+1):
        q = array[t,np.isfinite(array[t])]
        rows.append({'source':source, 'age':t, 'n':len(q), 'mean':q.mean(), 'median':np.median(q),
                     'sd':q.std(), 'variance':q.var(), 'zero_fraction':np.mean(q==0), 'gini':gini(q),
                     'q95':np.quantile(q,.95), 'max':q.max()})
moments = pd.DataFrame(rows)
moments.to_csv(OUTPUT / 'annual_moments.csv', index=False)
display(moments.loc[moments.age.eq(Y)])

,source,age,n,mean,median,sd,variance,zero_fraction,gini,q95,max
20,empirical_available,20,591,6.547803,4.493036,7.238070,5.238966e+01,0.159052,0.546418,21.246465,4.830581e+01
41,empirical_complete,20,510,6.789832,4.649225,7.332616,5.376726e+01,0.141176,0.535770,21.640393,4.830581e+01
62,arw4,20,10000,6.666496,5.596899,5.065964,2.566399e+01,0.000000,0.407445,16.379233,4.087215e+01
83,lognormal_grw,20,10000,17713.294556,10.593780,390576.585219,1.525501e+11,0.095500,0.991649,6861.429471,2.153972e+07
104,ar1_s,20,10000,6.983392,3.510835,11.180250,1.249980e+02,0.025000,0.605467,25.000076,2.094559e+02
125,hurdle_ar1_s_p,20,10000,6.281482,4.350520,7.331811,5.375545e+01,0.156600,0.537994,19.307084,8.174495e+01
146,ar6_6,20,10000,5.737718,4.205519,6.185528,3.826076e+01,0.164900,0.517664,16.926881,6.759209e+01


In [5]:
# fixed complete empirical cohort; prefix-conditioned AR(6,6) is marked in every rank row
rows = []
for source, array in sources.items():
    cumulative = array.cumsum(axis=0)
    for measure, a in [('annual',array), ('cumulative',cumulative)]:
        for anchor in [0,5]:
            for t in range(anchor,Y+1):
                rows.append({'source':source, 'measure':measure, 'reference':f'age_{anchor}', 'age':t,
                             'rho':correlation(a[anchor],a[t]), 'n':a.shape[1],
                             'empirical_prefix':source=='ar6_6' and t<6})
        for lag in [1,3,5]:
            for t in range(lag,Y+1):
                rows.append({'source':source, 'measure':measure, 'reference':f'lag_{lag}', 'age':t,
                             'rho':correlation(a[t-lag],a[t]), 'n':a.shape[1],
                             'empirical_prefix':source=='ar6_6' and t<6})
ranks = pd.DataFrame(rows)
ranks.to_csv(OUTPUT / 'rank_correlations.csv', index=False)

In [6]:
# available-case empirical transitions; NaN breaks a zero spell
hazards, transitions = [], []
for source, array in {'empirical_available':Q, **sources}.items():
    zero_run = np.zeros(array.shape[1],dtype=int)
    for t in range(Y):
        valid = np.isfinite(array[t]) & np.isfinite(array[t+1])
        zero_run = np.where(np.isfinite(array[t]) & (array[t]==0),zero_run+1,0)
        for duration in np.unique(zero_run[valid & (zero_run>0)]):
            risk = valid & (zero_run==duration)
            hazards.append({'source':source, 'age':t, 'duration':duration, 'n_at_risk':risk.sum(),
                            'n_restart':np.sum(array[t+1,risk]>0)})
        q, nxt = array[t,valid], array[t+1,valid]
        active, next_active = q>0, nxt>0
        transitions.append({'source':source, 'age':t, 'n':len(q),
                            'p_dropout':np.mean(~next_active[active]), 'p_restart':np.mean(next_active[~active]) if (~active).any() else np.nan,
                            'mean_increment':np.mean(nxt-q), 'sd_increment':np.std(nxt-q),
                            'mean_log_increment':np.mean(np.log(nxt+EPS)-np.log(q+EPS))})
hazard = pd.DataFrame(hazards)
hazard['generated_transition'] = ~((hazard.source=='ar6_6') & (hazard.age<5))
hazard.to_csv(OUTPUT / 'restart_by_age_duration.csv', index=False)
hazard = hazard.loc[hazard.generated_transition].groupby(['source','duration'],as_index=False)[['n_at_risk','n_restart']].sum()
hazard['restart_probability'] = hazard.n_restart/hazard.n_at_risk
hazard.to_csv(OUTPUT / 'restart_hazard.csv', index=False)
pd.DataFrame(transitions).to_csv(OUTPUT / 'transitions.csv', index=False)

In [7]:
# distribution diagnostics: common complete horizon plus the historical last-observed-four comparison
windows = {'last_four':(Y-3,Y), 'generated_common':(6,Y), 'career_total':(0,Y)}
rows, qq_rows, max_rows = [], [], []
for source, array in sources.items():
    for window,(start,end) in windows.items():
        total = array[start:end+1].sum(axis=0)
        positive = total[total>0]
        log_total = np.log2(positive)
        theoretical, ordered = stats.probplot(log_total, dist='norm', fit=False)
        slope, intercept, r = stats.probplot(log_total, dist='norm')[1]
        rows.append({'source':source, 'window':window, 'n':len(total), 'n_positive':len(positive),
                     'mean_total':total.mean(), 'gini_total':gini(total), 'qq_r':r, 'qq_r2':r*r,
                     'prefix_included':source=='ar6_6' and start<6})
        probabilities = (np.arange(199)+.5)/199
        standardized = (log_total-log_total.mean())/log_total.std(ddof=1)
        for x,y in zip(stats.norm.ppf(probabilities),np.quantile(standardized,probabilities)):
            qq_rows.append({'source':source,'window':window,'normal_quantile':x,'observed_quantile':y})
    # first maximum on ties, recorded rather than randomized
    maximum_age = array.argmax(axis=0)
    for age in range(Y+1):
        max_rows.append({'source':source, 'age':age, 'fraction':np.mean(maximum_age==age)})
last_observed = []
for col in Q.T:
    observed = np.flatnonzero(np.isfinite(col))
    if len(observed)>=4 and np.all(np.diff(observed[-4:])==1):
        last_observed.append(col[observed[-4:]].sum())
last_observed = np.asarray(last_observed)
log_total = np.log2(last_observed[last_observed>0])
r = stats.probplot(log_total,dist='norm')[1][2]
rows.append({'source':'empirical_last_observed', 'window':'last_four', 'n':len(last_observed),
             'n_positive':len(log_total), 'mean_total':last_observed.mean(), 'gini_total':gini(last_observed),
             'qq_r':r, 'qq_r2':r*r, 'prefix_included':False})
pd.DataFrame(rows).to_csv(OUTPUT / 'distribution_summary.csv', index=False)
pd.DataFrame(qq_rows).to_csv(OUTPUT / 'qq_points.csv', index=False)
pd.DataFrame(max_rows).to_csv(OUTPUT / 'year_of_maximum.csv', index=False)
display(pd.DataFrame(rows).query("window == 'last_four'"))

,source,window,n,n_positive,mean_total,gini_total,qq_r,qq_r2,prefix_included
0,empirical_complete,last_four,510,500,28.127164,0.467120,0.987660,0.975473,False
3,arw4,last_four,10000,10000,26.897091,0.318369,0.995625,0.991270,False
6,lognormal_grw,last_four,10000,9868,60210.153430,0.991661,0.992836,0.985724,False
9,ar1_s,last_four,10000,9999,27.947797,0.507315,0.995776,0.991569,False
12,hurdle_ar1_s_p,last_four,10000,9855,25.129366,0.447935,0.965536,0.932259,False
15,ar6_6,last_four,10000,9648,23.469428,0.442298,0.967378,0.935821,False
18,empirical_last_observed,last_four,2054,2017,24.431151,0.437630,0.987691,0.975534,False


In [8]:
# lag decay is a descriptive fit; AR(6,6) coefficients stay unconstrained
params = pd.read_csv(INPUT / 'fit/ar6_6/parameters.csv')
rows = []
for margin in ['activity','positive']:
    values = params[margin+'_beta'].to_numpy()
    fit = least_squares(lambda v:v[0]*v[1]**np.arange(6)-values, [values[0],.65],
                        bounds=([-np.inf,0],[np.inf,1]))
    rows.append({'margin':margin, 'lag1_weight':fit.x[0], 'rho':fit.x[1], 'rmse':np.sqrt(np.mean(fit.fun**2))})
pd.DataFrame(rows).to_csv(OUTPUT / 'geometric_lag_fit.csv', index=False)
display(pd.DataFrame(rows))

,margin,lag1_weight,rho,rmse
0,activity,0.749410,0.709290,0.053507
1,positive,0.324465,0.627798,0.010984
